# move to goal pose calculations

Calculations for setting a target pose for the end effector of the SO-ARM-100 robotic arm.

- The move_to_goal action assumes that the target pose is provided in the `base_link` frame.
- The yaw of the arm to the target pose is determined by the rotation of the shoulder joint, as this is the only joint with this DoF in the SO-ARM-100.
- The yaw angle is calculated by considering the angle the arm must rotate from its neutral position (which is along the negative y-axis of the `base_link` frame `{b}`) to the target.  

Poses of entities in `pick_n_place.sdf` the example:

- bin: [0.45, 0.3, 1.041, 0, 0, 0]
- red: [0.6568, -0.15, 1.041, 0, 0, 0]
- grn: [0.547, -0.2598, 1.041, 0, 0, 0]
- blu: [0.397, -0.3, 1.041, 0, 0, 0]
- yel: [0.247, -0.2598, 1.041, 0, 0, 0]
- arm: [0.25, 0.0, 1.015, 0, 0, 1.570796]


In [131]:
import math
import numpy as np
from transforms3d import affines
from transforms3d import euler
from transforms3d import quaternions


In [132]:
class Pose3:
    def __init__(self, position = np.zeros(3), orientation = np.array([1, 0, 0, 0])):
        self.position = np.array(position)
        self.orientation = np.array(orientation)
        self.affines = None
        self.compose(self.position, self.orientation)

    def compose(self, position, orientation):
        self.affines = np.array(
            affines.compose(T=position, R=quaternions.quat2mat(orientation), Z=np.ones(3))
        )

    def __str__(self):
        return (f"{self.position}\n" + f"{self.orientation}\n" + f"{self.affines}")

    def inv(self):
        inv_affines = np.linalg.inv(self.affines)
        trzs = affines.decompose(inv_affines)
        inv_pos = trzs[0]
        inv_rot = trzs[1]
        inv_pose = Pose3(inv_pos, quaternions.mat2quat(inv_rot))
        return inv_pose
    
    def apply(self, vec = np.zeros(3)):
        vec4_in = np.pad(vec, (0, 1))
        vec4_out = np.matmul(self.affines, vec4_in)
        return vec4_out[0:3]

In [133]:
# world {w} poses of entities
w_X_bin = Pose3([0.45, 0.3, 1.041])
w_X_red = Pose3([0.6568, -0.15, 1.041])
w_X_grn = Pose3([0.547, -0.2598, 1.041])
w_X_blu = Pose3([0.397, -0.3, 1.041])
w_X_yel = Pose3([0.247, -0.2598, 1.041])
w_X_arm = Pose3([0.25, 0.0, 1.015], euler.euler2quat(0, 0, np.pi / 2))

# print(w_X_bin)
# print(w_X_red)
# print(w_X_grn)
# print(w_X_blu)
# print(w_X_yel)
# print(w_X_arm)

# {w} vectors from arm to targets
w_v_arm_arm = w_X_arm.position - w_X_arm.position
w_v_arm_bin = w_X_bin.position - w_X_arm.position
w_v_arm_red = w_X_red.position - w_X_arm.position
w_v_arm_grn = w_X_grn.position - w_X_arm.position
w_v_arm_blu = w_X_blu.position - w_X_arm.position
w_v_arm_yel = w_X_yel.position - w_X_arm.position

# {w} vectors from shoulder to targets
w_v_arm_shd = np.array([0.0452, 0.0, 0.0165])
w_v_shd_bin = w_v_arm_bin - w_v_arm_shd
w_v_shd_red = w_v_arm_red - w_v_arm_shd
w_v_shd_grn = w_v_arm_grn - w_v_arm_shd
w_v_shd_blu = w_v_arm_blu - w_v_arm_shd
w_v_shd_yel = w_v_arm_yel - w_v_arm_shd

print("{w} vector: arm to shd")
print(w_v_arm_shd)

print("{w} vectors: arm to target")
# print(w_v_arm_bin)
# print(w_v_arm_red)
print(w_v_arm_grn)
# print(w_v_arm_blu)
# print(w_v_arm_yel)

print("{w} vectors: shd to target")
# print(w_v_shd_bin)
# print(w_v_shd_red)
print(w_v_shd_grn)
# print(w_v_shd_blu)
# print(w_v_shd_yel)

# use pad to convert vector3 into homogeneous form
# print()
# print(w_v_arm_red)
# print(np.pad(w_v_arm_red, (0, 1)))

{w} vector: arm to shd
[0.0452 0.     0.0165]
{w} vectors: arm to target
[ 0.297  -0.2598  0.026 ]
{w} vectors: shd to target
[ 0.2518 -0.2598  0.0095]


In [134]:
# transform to the arm base link {b}
w_X_b = w_X_arm
# b_X_w = np.linalg.inv(w_X_b.affines)
b_X_w = w_X_b.inv()

# this we expect to be identically zero
b_v_arm_arm = b_X_w.apply(w_v_arm_arm)
print(b_v_arm_arm)

# {b} vectors from arm to targets
b_v_arm_bin = b_X_w.apply(w_v_arm_bin)
b_v_arm_red = b_X_w.apply(w_v_arm_red)
b_v_arm_grn = b_X_w.apply(w_v_arm_grn)
b_v_arm_blu = b_X_w.apply(w_v_arm_blu)
b_v_arm_yel = b_X_w.apply(w_v_arm_yel)

# {b} vectors from shd to targets
b_v_shd_bin = b_X_w.apply(w_v_shd_bin)
b_v_shd_red = b_X_w.apply(w_v_shd_red)
b_v_shd_grn = b_X_w.apply(w_v_shd_grn)
b_v_shd_blu = b_X_w.apply(w_v_shd_blu)
b_v_shd_yel = b_X_w.apply(w_v_shd_yel)

print("{b} vectors: arm to target")
# print(b_v_arm_bin)
# print(b_v_arm_red)
print(b_v_arm_grn)
# print(b_v_arm_blu)
# print(b_v_arm_yel)

print("{b} vectors: shd to target")
# print(b_v_shd_bin)
# print(b_v_shd_red)
print(b_v_shd_grn)
# print(b_v_shd_blu)
# print(b_v_shd_yel)


[0. 0. 0.]
{b} vectors: arm to target
[-0.2598 -0.297   0.026 ]
{b} vectors: shd to target
[-0.2598 -0.2518  0.0095]


In [135]:
# yaw angles: b_v_unit_y is a unit reference vector along {b} -ve y-axis 
b_v_unit_x = np.array([1, 0, 0])
b_v_unit_y = np.array([0, -1, 0])

In [136]:
proj_x_bin = np.dot(b_v_shd_bin / np.linalg.norm(b_v_shd_bin), b_v_unit_x)
proj_y_bin = np.dot(b_v_shd_bin / np.linalg.norm(b_v_shd_bin), b_v_unit_y)
theta_z_bin = np.atan2(proj_x_bin, proj_y_bin)

proj_x_red = np.dot(b_v_shd_red / np.linalg.norm(b_v_shd_red), b_v_unit_x)
proj_y_red = np.dot(b_v_shd_red / np.linalg.norm(b_v_shd_red), b_v_unit_y)
theta_z_red = np.atan2(proj_x_red, proj_y_red)

proj_x_grn = np.dot(b_v_shd_grn / np.linalg.norm(b_v_shd_grn), b_v_unit_x)
proj_y_grn = np.dot(b_v_shd_grn / np.linalg.norm(b_v_shd_grn), b_v_unit_y)
theta_z_grn = np.atan2(proj_x_grn, proj_y_grn)

proj_x_blu = np.dot(b_v_shd_blu / np.linalg.norm(b_v_shd_blu), b_v_unit_x)
proj_y_blu = np.dot(b_v_shd_blu / np.linalg.norm(b_v_shd_blu), b_v_unit_y)
theta_z_blu = np.atan2(proj_x_blu, proj_y_blu)

proj_x_yel = np.dot(b_v_shd_yel / np.linalg.norm(b_v_shd_yel), b_v_unit_x)
proj_y_yel = np.dot(b_v_shd_yel / np.linalg.norm(b_v_shd_yel), b_v_unit_y)
theta_z_yel = np.atan2(proj_x_yel, proj_y_yel)

# print(f"{np.degrees(theta_z_bin):.2f}")
# print(f"{np.degrees(theta_z_red):.2f}")
print(f"{np.degrees(theta_z_grn):.2f}")
# print(f"{np.degrees(theta_z_blu):.2f}")
# print(f"{np.degrees(theta_z_yel):.2f}")


-45.90
